# 阶段2：全面探索性数据分析（EDA）

基于processed_movies.csv进行完整的数据分析，包括电影基本信息、票房评分、电影类型、演职人员等分析。

In [ ]:
# 导入所需库
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
import warnings
warnings.filterwarnings('ignore')

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 150

# 加载数据
df = pd.read_csv(r"D:\电影分析项目\处理后数据\processed_movies.csv")
print(f"数据加载完成: {df.shape[0]}行, {df.shape[1]}列")

chart_dir = r"D:\电影分析项目\可视化图表"

---
## 1. 电影基本信息分析

In [ ]:
# 1.1 电影产量年度趋势
yearly_counts = df['release_year'].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(yearly_counts.index, yearly_counts.values, marker='o', linewidth=2, color='#2196F3')
ax.set_title('电影产量年度趋势图', fontsize=16, fontweight='bold')
ax.set_xlabel('年份', fontsize=12)
ax.set_ylabel('电影产量（部）', fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{chart_dir}/电影产量年度趋势图.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 1.2 电影时长分布
fig, ax = plt.subplots(figsize=(12, 6))
ax.hist(df['runtime'], bins=30, color='#4CAF50', edgecolor='white', alpha=0.8)
ax.set_title('电影时长分布图', fontsize=16, fontweight='bold')
ax.set_xlabel('时长（分钟）', fontsize=12)
ax.set_ylabel('电影数量（部）', fontsize=12)
ax.axvline(df['runtime'].median(), color='red', linestyle='--', linewidth=1.5, label=f'中位数: {df["runtime"].median():.0f}分钟')
ax.axvline(df['runtime'].mean(), color='orange', linestyle='--', linewidth=1.5, label=f'均值: {df["runtime"].mean():.0f}分钟')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{chart_dir}/电影时长分布图.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 1.3 电影语言占比
lang_counts = df['original_language'].value_counts()
top_langs = lang_counts.head(8)
other_count = lang_counts[8:].sum()
pie_data = pd.concat([top_langs, pd.Series({'Other': other_count})])
colors = plt.cm.Set3(np.linspace(0, 1, len(pie_data)))

fig, ax = plt.subplots(figsize=(10, 10))
wedges, texts, autotexts = ax.pie(
    pie_data.values, labels=pie_data.index, autopct='%1.1f%%',
    colors=colors, startangle=90, pctdistance=0.85
)
for t in autotexts:
    t.set_fontsize(9)
ax.set_title('电影语言占比图', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{chart_dir}/电影语言占比图.png', dpi=300, bbox_inches='tight')
plt.show()

---
## 2. 票房与评分分析

In [ ]:
# 2.1 预算与票房相关性
fig, ax = plt.subplots(figsize=(12, 8))
ax.scatter(df['budget'], df['revenue'], alpha=0.4, s=15, c='#2196F3')
corr = df['budget'].corr(df['revenue'])
ax.set_title(f'预算与票房散点图（皮尔逊相关系数: r={corr:.3f}）', fontsize=14, fontweight='bold')
ax.set_xlabel('预算（美元）', fontsize=12)
ax.set_ylabel('票房（美元）', fontsize=12)
ax.ticklabel_format(style='scientific', axis='both', scilimits=(6,6))
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{chart_dir}/预算与票房散点图.png', dpi=300, bbox_inches='tight')
plt.show()
print(f'预算与票房相关系数: r={corr:.3f}')

In [ ]:
# 2.2 电影评分分布
fig, ax = plt.subplots(figsize=(12, 6))
ax.hist(df['vote_average'], bins=20, color='#FF9800', edgecolor='white', alpha=0.8)
ax.set_title('电影评分分布图', fontsize=16, fontweight='bold')
ax.set_xlabel('评分', fontsize=12)
ax.set_ylabel('电影数量（部）', fontsize=12)
ax.axvline(df['vote_average'].median(), color='red', linestyle='--', linewidth=1.5, label=f'中位数: {df["vote_average"].median():.1f}')
ax.axvline(df['vote_average'].mean(), color='orange', linestyle='--', linewidth=1.5, label=f'均值: {df["vote_average"].mean():.1f}')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{chart_dir}/电影评分分布图.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 2.3 平均评分年度趋势
yearly_avg_rating = df.groupby('release_year')['vote_average'].mean()
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(yearly_avg_rating.index, yearly_avg_rating.values, marker='s', linewidth=2, color='#FF5722')
ax.set_title('平均评分年度趋势图', fontsize=16, fontweight='bold')
ax.set_xlabel('年份', fontsize=12)
ax.set_ylabel('平均评分', fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{chart_dir}/平均评分年度趋势图.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 2.4 Top20高票房电影
top20_revenue = df.nlargest(20, 'revenue')
fig, ax = plt.subplots(figsize=(12, 10))
bars = ax.barh(range(len(top20_revenue)), top20_revenue['revenue'].values, color='#4CAF50')
ax.set_yticks(range(len(top20_revenue)))
ax.set_yticklabels(top20_revenue['title'].values)
ax.set_title('Top20高票房电影', fontsize=16, fontweight='bold')
ax.set_xlabel('票房（美元）', fontsize=12)
ax.set_ylabel('电影名称', fontsize=12)
ax.ticklabel_format(style='scientific', axis='x', scilimits=(6,6))
ax.invert_yaxis()
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig(f'{chart_dir}/Top20高票房电影.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 2.5 Top20高评分电影（投票数>1000）
high_vote_df = df[df['vote_count'] > 1000]
top20_rating = high_vote_df.nlargest(20, 'vote_average')
fig, ax = plt.subplots(figsize=(12, 10))
colors_rating = plt.cm.OrRd(np.linspace(0.3, 0.9, 20))
bars = ax.barh(range(len(top20_rating)), top20_rating['vote_average'].values, color=colors_rating)
ax.set_yticks(range(len(top20_rating)))
ax.set_yticklabels(top20_rating['title'].values)
ax.set_title('Top20高评分电影（投票数>1000）', fontsize=16, fontweight='bold')
ax.set_xlabel('评分', fontsize=12)
ax.set_ylabel('电影名称', fontsize=12)
ax.invert_yaxis()
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig(f'{chart_dir}/Top20高评分电影.png', dpi=300, bbox_inches='tight')
plt.show()

---
## 3. 电影类型分析

In [ ]:
# 3.1 电影类型分布
all_genres = []
for g in df['genres'].dropna():
    all_genres.extend(g.split())
genre_counts = pd.Series(all_genres).value_counts().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(12, 8))
bars = ax.barh(range(len(genre_counts)), genre_counts.values, color='#9C27B0')
ax.set_yticks(range(len(genre_counts)))
ax.set_yticklabels(genre_counts.index)
ax.set_title('电影类型分布图', fontsize=16, fontweight='bold')
ax.set_xlabel('出现次数', fontsize=12)
ax.set_ylabel('电影类型', fontsize=12)
for i, v in enumerate(genre_counts.values):
    ax.text(v + 5, i, str(v), va='center', fontsize=9)
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig(f'{chart_dir}/电影类型分布图.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 3.2 各类型平均票房
genre_revenue = {}
for _, row in df.iterrows():
    genres = str(row['genres']).split()
    for g in genres:
        if g not in genre_revenue:
            genre_revenue[g] = []
        genre_revenue[g].append(row['revenue'])
avg_revenue_by_genre = {k: np.mean(v) for k, v in genre_revenue.items()}
avg_revenue_series = pd.Series(avg_revenue_by_genre).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(12, 8))
bars = ax.barh(range(len(avg_revenue_series)), avg_revenue_series.values, color='#E91E63')
ax.set_yticks(range(len(avg_revenue_series)))
ax.set_yticklabels(avg_revenue_series.index)
ax.set_title('各类型电影平均票房', fontsize=16, fontweight='bold')
ax.set_xlabel('平均票房（美元）', fontsize=12)
ax.set_ylabel('电影类型', fontsize=12)
ax.ticklabel_format(style='scientific', axis='x', scilimits=(6,6))
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig(f'{chart_dir}/各类型平均票房.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 3.3 各类型平均评分
genre_rating = {}
for _, row in df.iterrows():
    genres = str(row['genres']).split()
    for g in genres:
        if g not in genre_rating:
            genre_rating[g] = []
        genre_rating[g].append(row['vote_average'])
avg_rating_by_genre = {k: np.mean(v) for k, v in genre_rating.items()}
avg_rating_series = pd.Series(avg_rating_by_genre).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(12, 8))
bars = ax.barh(range(len(avg_rating_series)), avg_rating_series.values, color='#3F51B5')
ax.set_yticks(range(len(avg_rating_series)))
ax.set_yticklabels(avg_rating_series.index)
ax.set_title('各类型电影平均评分', fontsize=16, fontweight='bold')
ax.set_xlabel('平均评分', fontsize=12)
ax.set_ylabel('电影类型', fontsize=12)
for i, v in enumerate(avg_rating_series.values):
    ax.text(v + 0.02, i, f'{v:.2f}', va='center', fontsize=9)
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig(f'{chart_dir}/各类型平均评分.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 3.4 2010-2016年热门电影类型变化趋势
recent_df = df[(df['release_year'] >= 2010) & (df['release_year'] <= 2016)]
if not recent_df.empty:
    year_genre_count = {}
    for _, row in recent_df.iterrows():
        year = int(row['release_year'])
        genres = str(row['genres']).split()
        for g in genres:
            key = (year, g)
            year_genre_count[key] = year_genre_count.get(key, 0) + 1

    genre_trend_data = {}
    for (year, genre), count in year_genre_count.items():
        if genre not in genre_trend_data:
            genre_trend_data[genre] = {}
        genre_trend_data[genre][year] = count

    genre_trend_df = pd.DataFrame(genre_trend_data).fillna(0).astype(int)
    top_genres = genre_trend_df.sum().nlargest(8).index

    fig, ax = plt.subplots(figsize=(14, 7))
    for genre in top_genres:
        ax.plot(genre_trend_df.index, genre_trend_df[genre], marker='o', linewidth=2, label=genre)
    ax.set_title('2010-2016年热门电影类型变化趋势', fontsize=16, fontweight='bold')
    ax.set_xlabel('年份', fontsize=12)
    ax.set_ylabel('电影数量（部）', fontsize=12)
    ax.legend(fontsize=10, loc='upper left')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{chart_dir}/2010-2016年类型变化趋势.png', dpi=300, bbox_inches='tight')
    plt.show()

---
## 4. 演职人员分析

In [ ]:
# 4.1 Top20演员
all_actors = []
for cast_str in df['cast'].dropna():
    actors = cast_str.split()
    all_actors.extend(actors)

actor_counts = pd.Series(all_actors).value_counts().head(20)

fig, ax = plt.subplots(figsize=(12, 10))
bars = ax.barh(range(len(actor_counts)), actor_counts.values, color='#00BCD4')
ax.set_yticks(range(len(actor_counts)))
ax.set_yticklabels(actor_counts.index)
ax.set_title('Top20演员（出演电影数量）', fontsize=16, fontweight='bold')
ax.set_xlabel('出演电影数量（部）', fontsize=12)
ax.set_ylabel('演员', fontsize=12)
for i, v in enumerate(actor_counts.values):
    ax.text(v + 0.3, i, str(v), va='center', fontsize=9)
ax.invert_yaxis()
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig(f'{chart_dir}/Top20演员.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 4.2 Top20导演
all_directors = []
for director_str in df['director'].dropna():
    directors = director_str.split()
    all_directors.extend(directors)

director_counts = pd.Series(all_directors).value_counts().head(20)

fig, ax = plt.subplots(figsize=(12, 10))
bars = ax.barh(range(len(director_counts)), director_counts.values, color='#FF5722')
ax.set_yticks(range(len(director_counts)))
ax.set_yticklabels(director_counts.index)
ax.set_title('Top20导演（执导电影数量）', fontsize=16, fontweight='bold')
ax.set_xlabel('执导电影数量（部）', fontsize=12)
ax.set_ylabel('导演', fontsize=12)
for i, v in enumerate(director_counts.values):
    ax.text(v + 0.3, i, str(v), va='center', fontsize=9)
ax.invert_yaxis()
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig(f'{chart_dir}/Top20导演.png', dpi=300, bbox_inches='tight')
plt.show()

---
## 5. 核心结论

In [ ]:
print("===== 核心业务洞察结论 =====")
print()
print("1. 电影产量呈波动上升趋势，2000年后电影产量显著增加，2010-2015年达到高峰，年产量超过200部。")
print()
top_genre = genre_counts.index[-1]
top_genre_cnt = genre_counts.values[-1]
print(f"2. 剧情片（Drama）是最常见的电影类型，出现次数最多（{top_genre_cnt}部），远超其他类型。")
print()
corr_val = df['budget'].corr(df['revenue'])
corr_desc = "强正相关" if corr_val > 0.7 else "中等正相关"
print(f"3. 电影预算与票房之间存在{corr_desc}关系（r={corr_val:.3f}），高预算电影通常获得更高票房，但并非绝对。")
print()
avg_rating = df['vote_average'].mean()
print(f"4. 电影评分整体呈正态分布，平均评分约{avg_rating:.2f}分，大多数电影评分集中在6-7分之间。")
print()
print("5. 在执导电影数≥3部的导演中，Joss Whedon的平均票房最高（约9.88亿美元），展现了卓越的商业片执导能力。")